In [ ]:
import os
import zipfile
import urllib.request

# Paths
ZIP_PATH = "/kaggle/working/s1.zip"
EXTRACT_PATH = "/kaggle/working/GRID"

# Download GRID S1
url = "https://zenodo.org/records/3625687/files/s1.zip?download=1"

if not os.path.exists(ZIP_PATH):
    print("Downloading S1...")
    urllib.request.urlretrieve(url, ZIP_PATH)
    print("Download complete.")
else:
    print("s1.zip already exists.")

# Extract
if not os.path.exists(EXTRACT_PATH):
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_PATH)
    print("Extraction complete.")
else:
    print("GRID folder already exists.")

print("ZIP:", ZIP_PATH)
print("DATA:", EXTRACT_PATH)

In [ ]:
import os
import glob

print("Checkpoint folder exists:",
      os.path.exists("/kaggle/working/models"))

print("\nCheckpoint files:")
for f in glob.glob("/kaggle/working/models/*"):
    print(f)

In [ ]:
import os
import cv2
import tensorflow as tf
import numpy as np

# S1 dataset
DATA_DIR = "/kaggle/working/GRID/s1"

# Check dataset
videos = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith(".mpg")
]

print("Number of videos:", len(videos))
print("First video:", videos[0])

In [ ]:
def load_video(path):
    cap = cv2.VideoCapture(path)
    frames = []

    for _ in range(int(cap.get(cv2.CAP_PROP_FRAME_COUNT))):
        ret, frame = cap.read()

        if not ret:
            break

        # Convert to grayscale
        frame = tf.image.rgb_to_grayscale(frame)

        # Crop mouth region
        frame = frame[190:236, 80:220, :]

        frames.append(frame)

    cap.release()

    # Normalize
    frames = tf.stack(frames)
    mean = tf.math.reduce_mean(frames)
    std = tf.math.reduce_std(tf.cast(frames, tf.float32))

    return tf.cast((frames - mean), tf.float32) / std

In [ ]:
import glob

alignments = glob.glob("/kaggle/working/GRID/**/*.align", recursive=True)

print("Alignment files found:", len(alignments))

if alignments:
    print("Example:", alignments[0])
else:
    print("No .align files found.")

In [ ]:
import os
import zipfile
import urllib.request

ALIGN_ZIP = "/kaggle/working/alignments.zip"
ALIGN_DIR = "/kaggle/working/GRID/alignments"

url = "https://zenodo.org/records/3625687/files/alignments.zip?download=1"

if not os.path.exists(ALIGN_ZIP):
    print("Downloading alignments...")
    urllib.request.urlretrieve(url, ALIGN_ZIP)
    print("Download complete.")
else:
    print("alignments.zip already exists.")

if not os.path.exists(ALIGN_DIR):
    print("Extracting alignments...")
    with zipfile.ZipFile(ALIGN_ZIP, "r") as zip_ref:
        zip_ref.extractall(ALIGN_DIR)
    print("Extraction complete.")
else:
    print("Alignments already extracted.")

print("Done.")

In [ ]:
import os
import glob

VIDEO_DIR = "/kaggle/working/GRID/s1"
ALIGN_DIR = "/kaggle/working/GRID/alignments/s1"

videos = sorted(glob.glob(os.path.join(VIDEO_DIR, "*.mpg")))
alignments = sorted(glob.glob(os.path.join(ALIGN_DIR, "*.align")))

print("Videos:", len(videos))
print("Alignments:", len(alignments))

video_names = {os.path.splitext(os.path.basename(x))[0] for x in videos}
align_names = {os.path.splitext(os.path.basename(x))[0] for x in alignments}

print("Matching pairs:", len(video_names & align_names))

In [ ]:
vocab = [x for x in "abcdefghijklmnopqrstuvwxyz'?!123456789 "]

char_to_num = tf.keras.layers.StringLookup(
    vocabulary=vocab,
    oov_token=""
)

num_to_char = tf.keras.layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(),
    oov_token="",
    invert=True
)

print("Vocabulary size:", char_to_num.vocabulary_size())

In [ ]:
def load_alignments(path):
    with open(path, "r") as f:
        lines = f.readlines()

    tokens = []

    for line in lines:
        line = line.split()

        if line[2] != "sil":
            tokens = [*tokens, " ", line[2]]

    return char_to_num(
        tf.reshape(
            tf.strings.unicode_split(
                tokens,
                input_encoding="UTF-8"
            ),
            (-1)
        )
    )[1:]

In [ ]:
def load_data(path):
    path = bytes.decode(path.numpy())

    file_name = os.path.splitext(os.path.basename(path))[0]

    video_path = os.path.join(
        VIDEO_DIR,
        f"{file_name}.mpg"
    )

    alignment_path = os.path.join(
        ALIGN_DIR,
        f"{file_name}.align"
    )

    frames = load_video(video_path)
    alignments = load_alignments(alignment_path)

    return frames, alignments

In [ ]:
def mappable_function(path):
    result = tf.py_function(
        load_data,
        [path],
        (tf.float32, tf.int64)
    )

    return result

In [ ]:
data = tf.data.Dataset.list_files(
    "/kaggle/working/GRID/s1/*.mpg"
)

data = data.shuffle(
    500,
    reshuffle_each_iteration=False
)

data = data.map(mappable_function)

data = data.padded_batch(
    2,
    padded_shapes=([75, None, None, None], [40])
)

data = data.prefetch(tf.data.AUTOTUNE)

print("Dataset pipeline created.")

In [ ]:
train = data.take(450)
test = data.skip(450)

print("Train batches:", 450)
print("Test batches:", "remaining")

In [ ]:
val = data.as_numpy_iterator().next()

print("Video batch shape:", val[0].shape)
print("Alignment batch shape:", val[1].shape)

In [ ]:
import glob
import os

alignments = glob.glob(
    "/kaggle/working/GRID/**/*.align",
    recursive=True
)

print("Total alignment files:", len(alignments))

for x in alignments[:10]:
    print(x)

In [ ]:
ALIGN_DIR = "/kaggle/working/GRID/alignments/alignments/s1"

alignments = glob.glob(
    os.path.join(ALIGN_DIR, "*.align")
)

print("S1 alignment files:", len(alignments))
print("Example:", alignments[0])

In [ ]:
sample_video = videos[0]

frames, alignment = load_data(
    tf.convert_to_tensor(sample_video)
)

print("Video:", os.path.basename(sample_video))
print("Frames shape:", frames.shape)
print("Alignment shape:", alignment.shape)
print("Text:", tf.strings.reduce_join(num_to_char(alignment)).numpy().decode())

In [ ]:
data = tf.data.Dataset.list_files(
    "/kaggle/working/GRID/s1/*.mpg"
)

data = data.shuffle(
    500,
    reshuffle_each_iteration=False
)

data = data.map(mappable_function)

data = data.padded_batch(
    2,
    padded_shapes=([75, None, None, None], [40])
)

data = data.prefetch(tf.data.AUTOTUNE)

train = data.take(450)
test = data.skip(450)

print("Dataset pipeline ready.")

In [ ]:
val = data.as_numpy_iterator().next()

print("Video batch shape:", val[0].shape)
print("Alignment batch shape:", val[1].shape)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(val[0][0][35])
plt.axis("off")
plt.show()

In [ ]:
print(
    tf.strings.reduce_join(
        [num_to_char(x) for x in val[1][0]]
    ).numpy().decode()
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv3D, LSTM, Dense, Dropout, Bidirectional,
    MaxPool3D, Activation, TimeDistributed,
    Flatten
)

model = Sequential()

model.add(
    Conv3D(
        128,
        3,
        input_shape=(75, 46, 140, 1),
        padding="same"
    )
)
model.add(Activation("relu"))
model.add(MaxPool3D((1, 2, 2)))

model.add(Conv3D(256, 3, padding="same"))
model.add(Activation("relu"))
model.add(MaxPool3D((1, 2, 2)))

model.add(Conv3D(75, 3, padding="same"))
model.add(Activation("relu"))
model.add(MaxPool3D((1, 2, 2)))

model.add(TimeDistributed(Flatten()))

model.add(
    Bidirectional(
        LSTM(
            128,
            kernel_initializer="Orthogonal",
            return_sequences=True
        )
    )
)
model.add(Dropout(0.5))

model.add(
    Bidirectional(
        LSTM(
            128,
            kernel_initializer="Orthogonal",
            return_sequences=True
        )
    )
)
model.add(Dropout(0.5))

model.add(
    Dense(
        char_to_num.vocabulary_size() + 1,
        kernel_initializer="he_normal",
        activation="softmax"
    )
)

model.summary()

In [ ]:
def scheduler(epoch, lr):
    if epoch < 30:
        return float(lr)
    else:
        return float(lr * 0.9048374180359595)

In [ ]:
def CTCLoss(y_true, y_pred):
    batch_len = tf.cast(
        tf.shape(y_true)[0],
        dtype="int64"
    )

    input_length = tf.cast(
        tf.shape(y_pred)[1],
        dtype="int64"
    )

    label_length = tf.cast(
        tf.shape(y_true)[1],
        dtype="int64"
    )

    input_length = input_length * tf.ones(
        shape=(batch_len, 1),
        dtype="int64"
    )

    label_length = label_length * tf.ones(
        shape=(batch_len, 1),
        dtype="int64"
    )

    loss = tf.keras.backend.ctc_batch_cost(
        y_true,
        y_pred,
        input_length,
        label_length
    )

    return loss

In [ ]:
class ProduceExample(tf.keras.callbacks.Callback):
    def __init__(self, dataset) -> None:
        self.dataset = dataset.as_numpy_iterator()

    def on_epoch_end(self, epoch, logs=None) -> None:
        data = self.dataset.next()

        yhat = self.model.predict(data[0])

        decoded = tf.keras.backend.ctc_decode(
            yhat,
            [75, 75],
            greedy=False
        )[0][0].numpy()

        for x in range(len(yhat)):
            print(
                "Original:",
                tf.strings.reduce_join(
                    num_to_char(data[1][x])
                ).numpy().decode("utf-8")
            )

            print(
                "Prediction:",
                tf.strings.reduce_join(
                    num_to_char(decoded[x])
                ).numpy().decode("utf-8")
            )

            print("~" * 100)

In [ ]:
from tensorflow.keras.optimizers import Adam

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss=CTCLoss
)

print("Model compiled successfully.")

In [ ]:
checkpoint_callback = ModelCheckpoint(
    os.path.join("models", "checkpoint.weights.h5"),
    monitor="loss",
    save_weights_only=True
)

In [ ]:
from tensorflow.keras.callbacks import LearningRateScheduler

schedule_callback = LearningRateScheduler(scheduler)

In [ ]:
example_callback = ProduceExample(test)

In [ ]:
import tensorflow as tf

print(tf.config.list_physical_devices("GPU"))

In [ ]:
model.fit(
    train,
    validation_data=test,
    epochs=100,
    callbacks=[
        checkpoint_callback,
        schedule_callback,
        example_callback
    ]
)